# Technical note: Paths and normalisation

Myokit does a lot of reading from and writing to the filesystem.
Filesystem paths can be represented in different ways (relative, absolute, symlinks, platform-dependent styles).
This document discusses:

1. What kind of sanitisation / normalisation to do, and where
2. Other annoyances related to paths and the file system


## Normalising paths

To avoid doing this all the time or inconsistently, we need some kind of policy.

### Slash and backslash

Windows uses `\` as separator, but accepts `/`, so `/` always works.
However, using `pathlib` we still get platform-dependent results.

1. **Use the native version on each platform**
2. The only exception might be unit tests where a path is converted to a string. But even here we could generate the validation result based on the platform.

### Symlinks

Symlinks can be _resolved_ to a "real" path.

1. **Only ever resolve if testing whether two files are the same. This might never be a Myokit concern..**
2. If the user hands us a symlink, keep the symlink.

### Relative paths: not starting at the root

Paths starting without a `/` or a drive letter are _relative to the working directory_.

The working directory can change between successive runs of a program.
And, although less likely, between different calls to an object's methods.
Therefore,

1. **Whenever storing a path, in an object property or a file, make it absolute first.**
2. When using a path in a function, keep it as is.


### Relative paths: using . and ..

The operators `.` and `..` exist mostly to specify relative paths.
However, they can both be used in "absolute" paths in the sense that `/one/two/../` points exactly at the absolute path `/one`.

1. **Leave `.` and `..` in paths, unless there's some security concern about which paths can be accessed.**


### Relative paths with `..` and symlinks

This is where it gets messy.
If `/path/to/symlink` is a link to `/heres/the/target`, then `path/to/symlink/..` can be interpreted as pointing to either `/path/to` or `/heres/the`.
For this reason, pathlib does not eliminate `..` sections unless also resolving symlinks.

See https://docs.python.org/3/library/pathlib.html#comparison-to-the-os-and-os-path-modules

### Expanding the user directory

On linux in particular, `~` is shorthand for the user's home directory.
This exists solely _to reduce typing_.

1. **Only expand the user directory when accepting input in shell scripts.**


### Expanding environment variables.

No.

## Old and new methods

| Method                    | Resolves symlinks | Makes absolute | Removes . and .. |
|---------------------------|-------------------|----------------|------------------|
| `os.path.abspath()`       | No                | Yes            | Yes              |
| `pathlib.Path.absolute()` | No                | Yes            | **No**           |
| `os.path.realpath()`      | Yes               | Yes            | Yes              |
| `pathlib.Path.resolve()`  | Yes               | Yes            | Yes              |

#### os.path.abspath()
```
os.path.abspath('a/b')           --> /cwd/a/b
os.path.abspath('/c/d/..')       --> /c
os.path.abspath('/link-to-e/f)   --> /link-to-e/f
```

#### os.path.realpath()
```
os.path.realpath('a/b')          --> /cwd/a/b
os.path.realpath('/c/d/..')      --> /c
os.path.realpath('/link-to-e/f') --> /e/f
```

#### pathlib.Path.absolute()
```
Path('a/b').absolute()           --> /cwd/a/b
Path('/c/d/..').absolute()       --> /c/d/..
Path('/link-to-e/f').absolute()  --> /link-to-a/b
```

#### pathlib.Path.resolve()
```
Path('a/b').resolve()           --> /cwd/a/b
Path('/c/d/..').resolve()       --> /c
Path('/link-to-e/f').resolve()  --> /e/f
```

## User directory

Myokit stores configuration settings in the user directory, under `~/.config/myokit`.
This works in _typical_ linux, mac, and windows settings.
It is possible not to have a user directory.
This contingency is not taken into account (until somebody asks for it, perhaps).